# Робота з індивідуальним споживанням електроенергії домогосподарствами

Тут реалізовані необхідні дії з набором даних про індивідуальне споживання електроенергії домогосподарствами

Ініціалізуємо необхідні бібліотеки, змінні

In [11]:
import pandas as pd
import numpy as np
import timeit
import scipy

Створимо DataFrame з файлу та здійснимо Data Cleaning

In [2]:
def txt_to_df(file):
    df = pd.read_csv(file, sep=";", low_memory=False)

    df.replace("?", np.nan, inplace=True)
    df.dropna(inplace=True)

    df.drop(columns=["Voltage", "Global_reactive_power"], inplace=True)

    df["Global_active_power"] = pd.to_numeric(df["Global_active_power"])
    df["Global_intensity"] = pd.to_numeric(df["Global_intensity"])

    df["Sub_metering_1"] = pd.to_numeric(df["Sub_metering_1"])
    df["Sub_metering_2"] = pd.to_numeric(df["Sub_metering_2"])
    df["Sub_metering_3"] = pd.to_numeric(df["Sub_metering_3"])

    df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"])

    return df

df = txt_to_df("data/household_power_consumption.txt")
print(df.head(5))

         Date      Time  Global_active_power  Global_intensity  \
0  16/12/2006  17:24:00                4.216              18.4   
1  16/12/2006  17:25:00                5.360              23.0   
2  16/12/2006  17:26:00                5.374              23.0   
3  16/12/2006  17:27:00                5.388              23.0   
4  16/12/2006  17:28:00                3.666              15.8   

   Sub_metering_1  Sub_metering_2  Sub_metering_3            Datetime  
0             0.0             1.0            17.0 2006-12-16 17:24:00  
1             0.0             1.0            16.0 2006-12-16 17:25:00  
2             0.0             2.0            17.0 2006-12-16 17:26:00  
3             0.0             1.0            17.0 2006-12-16 17:27:00  
4             0.0             1.0            17.0 2006-12-16 17:28:00  


Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [3]:
def filter_power_over_5kw(data):
    return data[data["Global_active_power"] > 5]

print(filter_power_over_5kw(df))

               Date      Time  Global_active_power  Global_intensity  \
1        16/12/2006  17:25:00                5.360              23.0   
2        16/12/2006  17:26:00                5.374              23.0   
3        16/12/2006  17:27:00                5.388              23.0   
11       16/12/2006  17:35:00                5.412              23.2   
12       16/12/2006  17:36:00                5.224              22.4   
...             ...       ...                  ...               ...   
2069356  22/11/2010  18:40:00                5.408              23.6   
2069357  22/11/2010  18:41:00                5.528              24.6   
2071586  24/11/2010  07:50:00                5.172              22.0   
2071587  24/11/2010  07:51:00                5.750              24.6   
2072997  25/11/2010  07:21:00                5.074              21.4   

         Sub_metering_1  Sub_metering_2  Sub_metering_3            Datetime  
1                   0.0             1.0            16.0 2

Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

In [4]:
def filter_between_19_20(data):
    filtered = data[
        (data["Global_intensity"] >= 19) &
        (data["Global_intensity"] <= 20)
    ]

    return filtered[
        (filtered["Sub_metering_2"] + filtered["Sub_metering_1"]) >
        (filtered["Sub_metering_3"])
    ]

print(filter_between_19_20(df))

               Date      Time  Global_active_power  Global_intensity  \
45       16/12/2006  18:09:00                4.464              19.0   
460      17/12/2006  01:04:00                4.582              19.6   
464      17/12/2006  01:08:00                4.618              19.6   
475      17/12/2006  01:19:00                4.636              19.4   
476      17/12/2006  01:20:00                4.634              19.4   
...             ...       ...                  ...               ...   
2071589  24/11/2010  07:53:00                4.666              19.8   
2071590  24/11/2010  07:54:00                4.694              19.8   
2071591  24/11/2010  07:55:00                4.602              19.4   
2071592  24/11/2010  07:56:00                4.536              19.0   
2071593  24/11/2010  07:57:00                4.626              19.4   

         Sub_metering_1  Sub_metering_2  Sub_metering_3            Datetime  
45                  0.0            37.0            16.0 2

Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [5]:
def filter_random_mean(data):
    rand = data.sample(500000, replace=False)

    return rand[
        ["Sub_metering_1",
         "Sub_metering_2",
         "Sub_metering_3"]
    ].mean()

print(filter_random_mean(df))

Sub_metering_1    1.115508
Sub_metering_2    1.299010
Sub_metering_3    6.482198
dtype: float64


Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [6]:
def filter_evening(data):
    evening = data[data["Datetime"].dt.hour >= 18]

    filtered = evening[
        evening["Global_active_power"] > 6
    ]

    filtered = filtered[
        filtered["Sub_metering_2"] >
        filtered[["Sub_metering_1", "Sub_metering_3"]].max(axis=1)
    ]

    half = len(filtered) // 2

    first_half = filtered.iloc[:half:3]
    second_half = filtered.iloc[half::4]

    return pd.concat([first_half, second_half])

print(filter_evening(df))

               Date      Time  Global_active_power  Global_intensity  \
41       16/12/2006  18:05:00                6.052              26.2   
44       16/12/2006  18:08:00                6.308              27.0   
17494    28/12/2006  20:58:00                6.386              27.0   
17498    28/12/2006  21:02:00                8.088              34.4   
17501    28/12/2006  21:05:00                7.230              30.6   
...             ...       ...                  ...               ...   
2066466  20/11/2010  18:30:00                6.620              29.2   
2066470  20/11/2010  18:34:00                6.266              27.6   
2066474  20/11/2010  18:38:00                6.302              27.8   
2066478  20/11/2010  18:42:00                6.238              27.6   
2066482  20/11/2010  18:46:00                6.438              28.4   

         Sub_metering_1  Sub_metering_2  Sub_metering_3            Datetime  
41                  0.0            37.0            17.0 2

Пронормувати та стандартизувати вибраний датасет.

In [34]:
def normalization(df):
    cols = [
        "Global_active_power",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3"
    ]

    normalized_df = df.copy()

    for col in cols:
        normalized_df[col] = (
                (df[col] - df[col].min()) /
                (df[col].max() - df[col].min())
        )

    return normalized_df

def standardization(df):
    cols = [
        "Global_active_power",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3"
    ]

    standardized_df = df.copy()

    for col in cols:
        standardized_df[col] = (
                (df[col] - df[col].mean()) /
                df[col].std()
        )

    return standardized_df

print(normalization(df))
print("==============")
print(standardization(df))

               Date      Time  Global_active_power  Global_intensity  \
0        16/12/2006  17:24:00             0.374796              18.4   
1        16/12/2006  17:25:00             0.478363              23.0   
2        16/12/2006  17:26:00             0.479631              23.0   
3        16/12/2006  17:27:00             0.480898              23.0   
4        16/12/2006  17:28:00             0.325005              15.8   
...             ...       ...                  ...               ...   
2075254  26/11/2010  20:58:00             0.078762               4.0   
2075255  26/11/2010  20:59:00             0.078580               4.0   
2075256  26/11/2010  21:00:00             0.078037               3.8   
2075257  26/11/2010  21:01:00             0.077675               3.8   
2075258  26/11/2010  21:02:00             0.077494               3.8   

         Sub_metering_1  Sub_metering_2  Sub_metering_3            Datetime  \
0                   0.0          0.0125        0.548387 

Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [21]:
def pearson_correlation(df):
    pearson = df["Global_active_power"].corr(
        df["Global_intensity"],
        method="pearson"
    )

    return pearson

def spearman_correlation(df):
    spearman = df["Global_active_power"].corr(
        df["Global_intensity"],
        method="spearman"
    )
    return spearman

print(str(pearson_correlation(df)),str(spearman_correlation(df)))

0.9988886002095853 0.9953719367299743


Провести One Hot Encoding категоріального атрибута.

In [22]:
def one_hot_encoding(df):
    df["Hour"] = df["Datetime"].dt.hour

    df["Day_period"] = pd.cut(
        df["Hour"],
        bins=[0, 6, 12, 18, 24],
        labels=["night", "morning", "day", "evening"]
    )

    encoded = pd.get_dummies(df["Day_period"])

    df = pd.concat([df, encoded], axis=1)
    return df

print(one_hot_encoding(df))

               Date      Time  Global_active_power  Global_intensity  \
0        16/12/2006  17:24:00                4.216              18.4   
1        16/12/2006  17:25:00                5.360              23.0   
2        16/12/2006  17:26:00                5.374              23.0   
3        16/12/2006  17:27:00                5.388              23.0   
4        16/12/2006  17:28:00                3.666              15.8   
...             ...       ...                  ...               ...   
2075254  26/11/2010  20:58:00                0.946               4.0   
2075255  26/11/2010  20:59:00                0.944               4.0   
2075256  26/11/2010  21:00:00                0.938               3.8   
2075257  26/11/2010  21:01:00                0.934               3.8   
2075258  26/11/2010  21:02:00                0.932               3.8   

         Sub_metering_1  Sub_metering_2  Sub_metering_3            Datetime  \
0                   0.0             1.0            17.0 

Профілювання часу виконання здійснити за допомогою модуля timeit.

In [31]:
def time_profiling(df, times):
    print(str(timeit.timeit(lambda: filter_power_over_5kw(df), number=times)) + "\n" +
          str(timeit.timeit(lambda: filter_between_19_20(df), number=times)) + "\n" +
           str(timeit.timeit(lambda: filter_random_mean(df), number=times)) + "\n" +
            str(timeit.timeit(lambda: filter_evening(df), number=times)))

time_profiling(df, 5)

0.03322019999905024
0.048020100002759136
1.0183164000045508
0.48862109999754466
